# Collecting Heating Data
by Tom Bignell
September 2025

A project to see what data we can get about heating a home and what insights we can get from this data. 

In [71]:
import requests
import webbrowser
import time
import json
import os
from dotenv import load_dotenv
import pandas as pd

## Get Access

Our heating is controlled by a Tado system. Therefore we need to start by getting access to the Tado system. 
They have an API by which we can gain access. 

To start, save your Client ID in a .env file in the same location as this code. 

The cell below uses this Client ID to gain an acess token by which you can access the Tado system. You also need to input your username and password into the browser when it opens. 

The script performs a secure OAuth 2.0 device authorization flow with the Tado API, allowing a user to authenticate without entering credentials directly into the script. Once authorized, it retrieves user account information and stores access tokens locally for future use.

In [72]:
CLIENT_ID = os.getenv("TADO_CLIENT_ID")
DEVICE_AUTH_URL = "https://login.tado.com/oauth2/device_authorize"
TOKEN_URL = "https://login.tado.com/oauth2/token"
API_URL = "https://my.tado.com/api/v2/me"

# Step 1: Request device code
print("🔐 Requesting device code...")
response = requests.post(
    DEVICE_AUTH_URL,
    params={
        "client_id": CLIENT_ID,
        "scope": "offline_access"
    },
    headers={"Content-Type": "application/x-www-form-urlencoded"}
)
data = response.json()
device_code = data["device_code"]
user_code = data["user_code"]
verification_uri = data["verification_uri_complete"]

print(f"📲 Please authorize access by visiting:\n{verification_uri}")
webbrowser.open(verification_uri)

# Step 2: Poll for access token
print("⏳ Waiting for user authorization...")
while True:
    token_response = requests.post(
        TOKEN_URL,
        data={
            "client_id": CLIENT_ID,
            "grant_type": "urn:ietf:params:oauth:grant-type:device_code",
            "device_code": device_code
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"}
    )
    token_data = token_response.json()

    if "access_token" in token_data:
        access_token = token_data["access_token"]
        refresh_token = token_data["refresh_token"]
        print("✅ Access granted!")
        break
    elif token_data.get("error") == "authorization_pending":
        time.sleep(5)
    else:
        print(f"❌ Error: {token_data}")
        exit()

# Step 3: Call Tado API
print("📡 Fetching user info...")
api_response = requests.get(
    API_URL,
    headers={"Authorization": f"Bearer {access_token}"}
)

# Optional: Save refresh token
with open("tado_tokens.json", "w") as f:
    import json
    json.dump({
        "access_token": access_token,
        "refresh_token": refresh_token
    }, f)
    print("💾 Tokens saved to tado_tokens.json")


🔐 Requesting device code...
📲 Please authorize access by visiting:
https://login.tado.com/oauth2/device?user_code=DSNRJ5
⏳ Waiting for user authorization...
✅ Access granted!
📡 Fetching user info...
💾 Tokens saved to tado_tokens.json


## Get Home Id

Tado gives each home an id, I guess a user could have more than one home. 

The code below gets the first home id and name.

If a user has multiple houses we would have to introduce the code here to chose which house to run the code for.

In [73]:
# Load saved tokens
with open("tado_tokens.json", "r") as f:
    tokens = json.load(f)

access_token = tokens["access_token"]

# Request user information
response = requests.get(
    "https://my.tado.com/api/v2/me",
    headers={"Authorization": f"Bearer {access_token}"}
)

# Save home id
home_id = json.loads(response.content).get('homes')[0].get('id')
home_name = json.loads(response.content).get('homes')[0].get('name')

print(f"💾 Home ID saved for:", home_name)

💾 Home ID saved for: Sylvia Avenue


## Get Room ID

Tado then assigns the different room heating setups and hot water different options.

In this cas we have one room and hot water, so Room 1 is 2 and Hot Water is 0.

In [74]:
# Request home information
zones_url = f"https://my.tado.com/api/v2/homes/{home_id}/zones"
zones_response = requests.get(zones_url, headers={"Authorization": f"Bearer {access_token}"})

# Save room and hot water id
room_id = zones_response.json()[0].get('id')
hotwater_id = zones_response.json()[1].get('id')

print(f"💾 Room and Hot Water ID saved for:", home_name)

💾 Room and Hot Water ID saved for: Sylvia Avenue


## Get the Data

We can then select a date to get the data for. From Tado we can collect data for:
- internal temperature
- internal humidity
- target heating temperature
- heating
- hot water
- weather

In [75]:
# Select a date to get the data for
date_str = "2025-01-01"

# Request room information for that date
room_url = f"https://my.tado.com/api/v2/homes/{home_id}/zones/{room_id}/dayReport?date={date_str}"
room_response = requests.get(room_url, headers={"Authorization": f"Bearer {access_token}"})

# Save the data as dictionaries
internal_temp = room_response.json().get('measuredData').get('insideTemperature').get('dataPoints')
humidity = room_response.json().get('measuredData').get('humidity').get('dataPoints')
target_temp = room_response.json().get('stripes').get('dataIntervals')
heating = room_response.json().get('callForHeat').get('dataIntervals')
hot_water = room_response.json().get('hotWaterProduction').get('dataIntervals')
weather = room_response.json().get('weather').get('condition').get('dataIntervals')

## Temperature Data
- Put the dictionary data into a datafame. 
- Turn the time into a datetime
- Extract the celcius value from the temperature values

In [76]:
#Turn dataPoints into a dataframe
df_internal_temp = pd.DataFrame(internal_temp)

#Convert the date and time to a date and time value
df_internal_temp['timestamp'] = pd.to_datetime(df_internal_temp['timestamp'])

#The value column contains both celsius and farenheit values, split this out
df_internal_temp = pd.concat([df_internal_temp.drop(["value"],axis=1), df_internal_temp["value"].apply(pd.Series)], axis=1)

#Remove the fahrenheit column
df_internal_temp = df_internal_temp.drop(["fahrenheit"], axis = 1)

#Rename the celsius column
df_internal_temp = df_internal_temp.rename(columns={"celsius":"internal_temperature_celsius", "timestamp":"time"})

df_internal_temp.head()

,time,internal_temperature_celsius
0,2024-12-31 23:45:00+00:00,17.43
1,2025-01-01 00:00:00+00:00,17.37
2,2025-01-01 00:15:00+00:00,17.30
3,2025-01-01 00:30:00+00:00,17.30
4,2025-01-01 00:45:00+00:00,17.26


## Humidity Data
- Put the dictionary data into a datafame. 
- Turn the time into a datetime
- Rename the values column

In [77]:
#Turn dataPoints into a dataframe
df_humidity = pd.DataFrame(humidity)

#Convert the date and time to a date and time value
df_humidity['timestamp'] = pd.to_datetime(df_humidity['timestamp'])

#Rename the celsius column
df_humidity = df_humidity.rename(columns={"value":"humidity_percentage", "timestamp":"time"})

df_humidity.head()

,time,humidity_percentage
0,2024-12-31 23:45:00+00:00,0.695
1,2025-01-01 00:00:00+00:00,0.695
2,2025-01-01 00:15:00+00:00,0.695
3,2025-01-01 00:30:00+00:00,0.695
4,2025-01-01 00:45:00+00:00,0.694


## Taget Temperature Data
- Put the dictionary data into a datafame. 
- Turn the time into a datetime
- Extract the celcius value from the temperature values
- Drop columns which are not required

In [78]:
#Turn dataPoints into a dataframe
df_target_temp = pd.DataFrame(target_temp)

#Convert the date and time to a date and time value
df_target_temp['from'] = pd.to_datetime(df_target_temp['from'])
df_target_temp['to'] = pd.to_datetime(df_target_temp['to'])

#The value column contains anpother dictionay, split this out
df_target_temp = pd.concat([df_target_temp.drop(["value"],axis=1), df_target_temp["value"].apply(pd.Series)], axis=1)
df_target_temp = pd.concat([df_target_temp.drop(["setting"],axis=1), df_target_temp["setting"].apply(pd.Series)], axis=1)
df_target_temp = pd.concat([df_target_temp.drop(["temperature"],axis=1), df_target_temp["temperature"].apply(pd.Series)], axis=1)

#Remove the fahrenheit, to, type and power columns
df_target_temp = df_target_temp.drop(["fahrenheit","to","stripeType","type","power"], axis = 1)

#Rename the celsius column
df_target_temp = df_target_temp.rename(columns={"celsius":"target_temperature_celsius", "from":"time"})

df_target_temp.head()

,time,target_temperature_celsius
0,2024-12-31 23:45:00+00:00,12.0
1,2025-01-01 06:30:00+00:00,18.0
2,2025-01-01 08:30:00+00:00,16.5
3,2025-01-01 10:30:00+00:00,18.0
4,2025-01-01 12:30:00+00:00,17.0


## Heating Data
- Put the dictionary data into a datafame. 
- Turn the time into a datetime
- Extract the heating level and rename the column
- Drop columns which are not required

In [79]:
#Turn dataPoints into a dataframe
df_heating = pd.DataFrame(heating)

#Convert the date and time to a date and time value
df_heating['from'] = pd.to_datetime(df_heating['from'])
df_heating['to'] = pd.to_datetime(df_heating['to'])

#Remove the to column
df_heating = df_heating.drop("to", axis = 1)

#Rename the value column
df_heating = df_heating.rename(columns={"value":"heating_level", "from":"time"})

df_heating.head()

,time,heating_level
0,2024-12-31 23:45:00+00:00,NONE
1,2025-01-01 06:33:20+00:00,HIGH
2,2025-01-01 08:30:16+00:00,LOW
3,2025-01-01 08:41:37+00:00,NONE
4,2025-01-01 10:30:00+00:00,LOW


## Hot Water Data
- Put the dictionary data into a datafame. 
- Turn the time into a datetime
- Extract the hot water indicator and rename the column
- Drop columns which are not required

In [80]:
#Turn dataPoints into a dataframe
df_hot_water = pd.DataFrame(hot_water)

#Convert the date and time to a date and time value
df_hot_water['from'] = pd.to_datetime(df_hot_water['from'])
df_hot_water['to'] = pd.to_datetime(df_hot_water['to'])

#Remove the to column
df_hot_water = df_hot_water.drop("to", axis = 1)

#Rename the value column
df_hot_water = df_hot_water.rename(columns={"value":"hot_water", "from":"time"})

df_hot_water.head()

,time,hot_water
0,2024-12-31 23:45:00+00:00,False
1,2025-01-01 07:00:00+00:00,True
2,2025-01-01 08:00:00+00:00,False
3,2025-01-01 18:00:00+00:00,True
4,2025-01-01 19:00:00+00:00,False


## Weather Data
- Put the dictionary data into a datafame. 
- Turn the time into a datetime
- Extract the outside weather and outside temperature and rename the column
- Drop columns which are not required

In [86]:
#Turn dataPoints into a dataframe
df_weather = pd.DataFrame(weather)

#Convert the date and time to a date and time value
df_weather['from'] = pd.to_datetime(df_weather['from'])
df_weather['to'] = pd.to_datetime(df_weather['to'])

#The value column contains anpother dictionay, split this out
df_weather = pd.concat([df_weather.drop(["value"],axis=1), df_weather["value"].apply(pd.Series)], axis=1)
df_weather = pd.concat([df_weather.drop(["temperature"],axis=1), df_weather["temperature"].apply(pd.Series)], axis=1)

#Remove the fahrenheit, to, type and power columns
df_weather = df_weather.drop(["fahrenheit","to"], axis = 1)

#Rename the celsius column
df_weather = df_weather.rename(columns={"celsius":"external_temperature_celsius", "from":"time"})

df_weather.head()

,time,state,external_temperature_celsius
0,2024-12-31 23:45:00+00:00,NIGHT_CLOUDY,11.81
1,2024-12-31 23:48:56+00:00,NIGHT_CLOUDY,11.81
2,2025-01-01 00:08:34+00:00,NIGHT_CLOUDY,11.81
3,2025-01-01 00:20:19+00:00,NIGHT_CLOUDY,11.86
4,2025-01-01 00:37:15+00:00,NIGHT_CLOUDY,11.86


## Merge the data together

Merge the dataframes together to get all the Tado data in one dataframe. Use an outer merge to get all the rows and join on the time column.

In [87]:
# Merge the intnernal temperature data with the humidity data to start creating a dataframe with all thermostat data in
df_thermostat = df_internal_temp.merge(df_humidity, how="outer", on="time")

# Merge the termostat data with the target temperature data
df_thermostat = df_thermostat.merge(df_target_temp, how="outer", on="time")

# Merge the termostat data with the heating data
df_thermostat = df_thermostat.merge(df_heating, how="outer", on="time")

# Merge the termostat data with the hot water data
df_thermostat = df_thermostat.merge(df_hot_water, how="outer", on="time")

# Merge the termostat data with the weather data
df_thermostat = df_thermostat.merge(df_weather, how="outer", on="time")

df_thermostat.head()

,time,internal_temperature_celsius,humidity_percentage,target_temperature_celsius,heating_level,hot_water,state,external_temperature_celsius
0,2024-12-31 23:45:00+00:00,17.43,0.695,12.0,NONE,False,NIGHT_CLOUDY,11.81
1,2024-12-31 23:48:56+00:00,NaN,NaN,NaN,NaN,NaN,NIGHT_CLOUDY,11.81
2,2025-01-01 00:00:00+00:00,17.37,0.695,NaN,NaN,NaN,NaN,NaN
3,2025-01-01 00:08:34+00:00,NaN,NaN,NaN,NaN,NaN,NIGHT_CLOUDY,11.81
4,2025-01-01 00:15:00+00:00,17.30,0.695,NaN,NaN,NaN,NaN,NaN


## NULLS

The dataframe contains nulls where individual dataframes were mergered and there was no data for a specific time. 

For target temperature, heating level, hot water and state, these all picked up when a change happened. So these could all be filled down.

For the internal temperature, humidity and external temperature columns we will need to interpolate based on time.

In [89]:
#Fill the NaN values with the value from above for target temperature, value, state and sunny
df_thermostat[["target_temperature_celsius","heating_level","hot_water","state"]]= df_thermostat[["target_temperature_celsius","heating_level","hot_water","state"]].fillna(method="ffill")

#Create a copy of the timestamp and columns to interpolate on
df_time = df_thermostat[["time","internal_temperature_celsius","humidity_percentage","external_temperature_celsius"]].copy()

#Set the index of the dataframe to match the timestamp
df_time = df_time.set_index(df_time['time']) 

#Drop the timestamp column
df_time = df_time.drop("time",axis=1)

#Interpolte on the data, based on the time, to find the missing values
df_time = df_time.interpolate(method='time')

#Copy the interpolated values over onto the main dataframe
df_thermostat[["internal_temperature_celsius","humidity_percentage","external_temperature_celsius"]] = df_time[["internal_temperature_celsius","humidity_percentage","external_temperature_celsius"]].values

df_thermostat.head()

/var/folders/ml/s2r5g8_x31l75sq2ql0l0qww0000gn/T/ipykernel_2819/4112815114.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_thermostat[["target_temperature_celsius","heating_level","hot_water","state"]]= df_thermostat[["target_temperature_celsius","heating_level","hot_water","state"]].fillna(method="ffill")


,time,internal_temperature_celsius,humidity_percentage,target_temperature_celsius,heating_level,hot_water,state,external_temperature_celsius
0,2024-12-31 23:45:00+00:00,17.430000,0.695,12.0,NONE,False,NIGHT_CLOUDY,11.810000
1,2024-12-31 23:48:56+00:00,17.414267,0.695,12.0,NONE,False,NIGHT_CLOUDY,11.810000
2,2025-01-01 00:00:00+00:00,17.370000,0.695,12.0,NONE,False,NIGHT_CLOUDY,11.810000
3,2025-01-01 00:08:34+00:00,17.330022,0.695,12.0,NONE,False,NIGHT_CLOUDY,11.810000
4,2025-01-01 00:15:00+00:00,17.300000,0.695,12.0,NONE,False,NIGHT_CLOUDY,11.837376
